# SRQ generalization M10 — controlled GACL adapter

This notebook checks inverse-RLS/primal equivalence and compares FP32, FP16, and P2B after every mini-batch on a fixed generalized CIFAR-100 training stream. It is train-only and is not an official end-to-end GACL reproduction.

In [ ]:
# Edit repository/path values only. Method, stream, thresholds, and gates are source-locked.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m10_cifar_features'
OUTPUT_DIR='/content/srq_m10_gacl_output'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and immutable source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def source_sha(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m10_gacl_train_only.json':'f3756aa078823a472740fe07a7898e32a04964b1ddf43862304e56013239e4d4',
 'tools/srq_generalization_m10.py':'02d1cd1787b57372ee2573fee0ad042931a61bd8f38a02382422589470076534',
 'methods/frontends/gacl.py':'e8337eea33c8404d3531421c825f86bd339129f033fa58b8d5f9aa2bcc065c30',
 'methods/frontends/__init__.py':'80cbc117f6112d278944fa05949eb982c3928142796fdcde9a1fd9b05615b7d9',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert source_sha(path)==expected,(path,source_sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m10_gacl_train_only.json'
RUNNER='tools/srq_generalization_m10.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE LOCK: PASS')

In [ ]:
# Local equation, stream, and backend gates before downloading data.
command=[sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_gacl_analytic_frontend.py','tests/test_srq_generalization_m10.py','tests/test_analytic_ridge_backend.py','tests/test_analytic_ridge_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M10 local correctness gate failed; return the complete traceback.'
print('M10 LOCAL GATES: PASS')

In [ ]:
# Download the locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out test features remain absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m10','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run four matched analytic paths; every path updates after every mini-batch.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M10 START: width=5000, fixed Si-Blurry N=50/M=10, batch=64, train-only.',flush=True)
print('This can take tens of minutes because P2B is decoded, QR-updated, and requantized after every mini-batch.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m10_results.json'
assert result_path.is_file(),'M10 failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M10_GACL_CONTROLLED_TRAIN_ONLY','M10 failed; preserve the result, do not relax gates, and do not inspect test accuracy.'

In [ ]:
# Human-readable task-boundary audit.
import pandas as pd
rows=[]
for record in result['records']:
    row={'task':record['task_id'],'updates':record['mini_batch_updates'],'new_classes':record['new_class_count_within_task'],'exposed_classes':record['previously_exposed_class_count_within_task'],'seen_classes':record['evaluation']['seen_class_count'],'inverse_primal_agreement':record['evaluation']['inverse_primal_prediction_agreement'],'p2b_primal_agreement':record['evaluation']['p2b_primal_prediction_agreement']}
    row.update({name+'_accuracy':value for name,value in record['evaluation']['accuracy_percent'].items()})
    rows.append(row)
display(pd.DataFrame(rows))

In [ ]:
# Export immutable evidence only; the sample-level feature cache is excluded.
bundle=Path('/content/srq_generalization_m10_gacl_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(Path(OUTPUT_DIR)/'m10_results.json',bundle/'m10_results.json')
shutil.copy2(CONFIG,bundle/'config.json')
shutil.copy2('docs/research/SRQ_GENERALIZATION_M10_RUNBOOK.md',bundle/'runbook.md')
manifest={'artifact':'srq_generalization_m10_gacl_train_only','uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_hashes':EXPECTED,'result_sha256':sha(Path(OUTPUT_DIR)/'m10_results.json')}
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)